# Lab 14B: Training mechanics and finding bugs

Exam questions often show a training loop and ask what happens: which gradients are computed, what a layer does in
training vs evaluation mode, how many parameters are trained, or what is wrong with the code. This notebook trains that.

**How to work:** as in Lab 14A – think about the question, run the cell, then read the explanation below it.
Runs on a CPU in a few minutes; no data is downloaded.

| Part | Topic | Learning goals |
|---|---|---|
| 1 | Autograd and `zero_grad()` | training code (P2), by hand (P4) |
| 2 | Training mode vs evaluation mode | training code (P2) |
| 3 | Transfer learning in code | training code, sizes (P2, P3) |
| 4 | Find the bug | diagnose and fix (P7) |
| 5 | Reading training curves | plots, diagnose (P6, P7) |

Each part starts with the formulas or rules you need.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
_ = torch.manual_seed(0)

---
## Part 1 – Autograd and `zero_grad()`

**Formulas.**
- **Chain rule:** if $f$ depends on $x$ through an intermediate value $u$, then
  $\frac{\partial f}{\partial x} = \frac{\partial f}{\partial u} \cdot \frac{\partial u}{\partial x}$ (upstream gradient × local gradient).
- Local gradients you need here: $\frac{\partial (ab)}{\partial a} = b$, $\frac{\partial (u + c)}{\partial c} = 1$, $\frac{d(u^2)}{du} = 2u$.
- **Gradient descent step:** $w \leftarrow w - \eta \, \frac{\partial L}{\partial w}$ ($\eta$ = learning rate). `loss.backward()` computes
  $\partial L / \partial w$ and **adds** it to `w.grad`; `optimizer.step()` does the update; `optimizer.zero_grad()` resets `w.grad`.

### Exercise 1.1 – Gradients of a small graph

**f = (a·b + c)². Compute ∂f/∂a, ∂f/∂b and ∂f/∂c by hand with the chain rule for a = 2, b = −3, c = 4.**

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-3.0, requires_grad=True)
c = torch.tensor(4.0, requires_grad=True)
u = a * b + c          # intermediate value
f = u ** 2
f.backward()           # fills a.grad, b.grad, c.grad
print(u.item(), f.item())
print(a.grad.item(), b.grad.item(), c.grad.item())

<details><summary><b>Show the answer</b></summary>

u = 2·(−3) + 4 = −2 and f = 4. ∂f/∂u = 2u = −4. Chain rule: ∂f/∂a = −4·b = 12, ∂f/∂b = −4·a = −8, ∂f/∂c = −4·1 = −4.
This is exactly what backpropagation does: the upstream gradient (−4) times the local gradient of each input.

</details>

### Exercise 1.2 – Gradients accumulate

**We call `backward()` three times. What is printed in the first loop, and what in the second?**

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
for step in range(3):
    loss = w ** 2
    loss.backward()
    print("without zeroing:", w.grad.item())

for step in range(3):
    w.grad = None              # this is what optimizer.zero_grad() does for every parameter
    loss = w ** 2
    loss.backward()
    print("with zeroing:   ", w.grad.item())

<details><summary><b>Show the answer</b></summary>

d(w²)/dw = 2w = 4. `backward()` **adds** the new gradient to `.grad`, so without zeroing you get 4, 8, 12. That is why a
training loop calls `optimizer.zero_grad()` before every `backward()`; with it, the gradient is 4 every time.

</details>

### Exercise 1.3 – `torch.no_grad()`

**What is printed?**

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
y1 = 3 * w
with torch.no_grad():
    y2 = 3 * w
print(y1.requires_grad, y2.requires_grad)

<details><summary><b>Show the answer</b></summary>

Inside `torch.no_grad()` PyTorch does not record the operations, so `y2` has no graph and cannot be back-propagated:
`True False`. This saves memory and time at test time.

</details>

---
## Part 2 – Training mode vs evaluation mode

`model.train()` and `model.eval()` switch layers such as `Dropout` and `BatchNorm` between two behaviours. A new model starts
in training mode.

**Formulas.**
- **Dropout** with probability $p$, training mode: each value is set to $0$ with probability $p$, otherwise multiplied by
  $\frac{1}{1-p}$ (so the expected value is unchanged). Evaluation mode: output = input.
- **BatchNorm**, per feature or channel: $y = \gamma \, \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$ (at the start $\gamma = 1$, $\beta = 0$).
  - Training mode: $\mu$ and $\sigma^2$ are the mean and the (biased) variance of the **current batch**.
  - After each training batch the running statistics are updated with `momentum` $m = 0.1$:
    $\hat\mu \leftarrow (1 - m)\,\hat\mu + m\,\mu_{batch}$, and likewise $\hat\sigma^2$ with the unbiased batch variance.
    They start at $\hat\mu = 0$ and $\hat\sigma^2 = 1$.
  - Evaluation mode: $\mu = \hat\mu$ and $\sigma^2 = \hat\sigma^2$ (the running statistics).

### Exercise 2.1 – Dropout

**What values can the output contain in training mode, and what in evaluation mode?**

In [ ]:
drop = nn.Dropout(p=0.2)
x = torch.ones(1, 1000)
drop.train()
print("train:", torch.unique(drop(x)))
drop.eval()
print("eval: ", torch.unique(drop(x)))

<details><summary><b>Show the answer</b></summary>

In training mode each value is set to 0 with probability 0.2, and the remaining values are multiplied by
1/(1 − 0.2) = 1.25, so the expected value stays 1. In evaluation mode dropout does nothing: all values are 1.

</details>

### Exercise 2.2 – BatchNorm in the two modes

A batch of 3 samples with 2 features (`momentum=0.1`, the running mean starts at 0).

**What is (a) the normalised first column in training mode and (b) the running mean after this one batch?**

In [ ]:
bn = nn.BatchNorm1d(num_features=2)       # gamma = 1, beta = 0 at the start
x = torch.tensor([[2.0, 10.0], [4.0, 20.0], [6.0, 30.0]])
bn.train()
print(bn(x))
print("running mean:", bn.running_mean)
print("running var: ", bn.running_var)
bn.eval()
print(bn(x))

<details><summary><b>Show the answer</b></summary>

(a) Training mode uses the statistics of **this batch**: the first column has mean 4 and variance 8/3, so
(2 − 4)/√(8/3) ≈ −1.22, then 0, then 1.22. The second column gives the same values. (b) The running mean is updated as
0.9·old + 0.1·batch mean = (0.4, 2.0); the running variance likewise from 1 towards the (unbiased) batch variance: (1.3, 10.9).
In evaluation mode the layer uses these running values instead, so the same input gives a different output. After one batch they
are still far from the data statistics – they are meant to be averaged over many batches.

</details>

### Exercise 2.3 – `no_grad()` is not `eval()`

**The model is still in training mode. Does dropout drop values inside `torch.no_grad()`?**

In [ ]:
drop = nn.Dropout(p=0.5)
drop.train()
with torch.no_grad():
    out = drop(torch.ones(1, 1000))
print((out == 0).any().item())

<details><summary><b>Show the answer</b></summary>

Yes: `True`. `torch.no_grad()` only switches off gradient recording; the layer behaviour is controlled by
`train()`/`eval()`. For testing you need **both**: `model.eval()` and `torch.no_grad()`.

</details>

---
## Part 3 – Transfer learning in code

We use a ResNet-18 with random weights (`weights=None`, so nothing is downloaded); the parameter counts are the same as for
the pretrained model.

**Rules.**
- A parameter is trained only if it has `requires_grad=True` (and is passed to the optimizer).
- A newly created layer always has `requires_grad=True`.
- A new head `Linear(D_in, n_classes)` has $D_{in} \cdot n_{classes} + n_{classes}$ parameters ($D_{in}$ = `model.fc.in_features`).

### Exercise 3.1 – Freeze, then replace the head

**How many parameters are trainable (`requires_grad=True`) at the end?**

In [ ]:
def trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

m = torchvision.models.resnet18(weights=None)
print("all parameters:", trainable(m))
print("input of fc:", m.fc.in_features)
for p in m.parameters():
    p.requires_grad = False                  # freeze everything
m.fc = nn.Linear(in_features=512, out_features=3)   # new head for 3 classes
print("trainable:", trainable(m))

<details><summary><b>Show the answer</b></summary>

A new layer is created with `requires_grad=True`, so only the new head is trained: 512·3 + 3 = 1,539 parameters.
The other 11.2 million are frozen: the backbone works as a fixed feature extractor.

</details>

### Exercise 3.2 – The order matters

**Same two steps, in the opposite order. How many parameters are trainable now?**

In [ ]:
m = torchvision.models.resnet18(weights=None)
m.fc = nn.Linear(in_features=512, out_features=3)
for p in m.parameters():
    p.requires_grad = False
print("trainable:", trainable(m))

<details><summary><b>Show the answer</b></summary>

0: the loop freezes **every** parameter of the model at that moment, including the new head. Training this model would
not change anything (and the optimizer would get no trainable parameters).

</details>

### Exercise 3.3 – Fine-tuning only the last block

**Now we unfreeze the last residual stage (`layer4`) as well. Roughly how many parameters are trained: thousands or millions?**

In [ ]:
m = torchvision.models.resnet18(weights=None)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(in_features=512, out_features=3)
for p in m.layer4.parameters():
    p.requires_grad = True
print("trainable:", trainable(m))
optimizer = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=1e-3)

<details><summary><b>Show the answer</b></summary>

About 8.4 million: `layer4` has the most channels (512) and holds most of the weights of a ResNet-18. Fine-tuning the last
block therefore needs more data than training only the head. Passing only the trainable parameters to the optimizer is common
practice; frozen parameters get no gradient anyway.

</details>

---
## Part 4 – Find the bug

**Formulas.** `nn.CrossEntropyLoss` takes raw scores (logits) $z$ and the true class $y$:
$L = -\log \frac{e^{z_y}}{\sum_j e^{z_j}}$. Guessing 50/50 between $C$ classes gives $L = \ln C$ (0.693 for 2 classes).
The predicted class is the argmax over the class dimension (`dim=1` for an output of shape $(N, C)$).

A small classification problem: points in 2-D, class 1 if x₁·x₂ > 0 (the points in the first and third quadrant), class 0
otherwise. A small MLP with dropout can learn this to about 95 % test accuracy. Run the next cell; it defines the data, the
model, the evaluation and a **correct** training loop. Each exercise then shows a version with **one bug**. Find it by
reading the code first, then run the cell and compare with the correct version.

In [ ]:
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 2, generator=g)
    y = (X[:, 0] * X[:, 1] > 0).long()          # labels 0 or 1
    return X, y

X_tr, y_tr = make_data(1000, seed=1)              # training set
X_te, y_te = make_data(1000, seed=2)              # test set

def make_model():
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.2),
                         nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
                         nn.Linear(64, 2))         # 2 class scores (logits)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def train(model, epochs=30, bs=50, lr=0.05):
    torch.manual_seed(1)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))           # shuffle every epoch
        for i in range(0, len(X_tr), bs):
            idx = perm[i:i + bs]
            xb, yb = X_tr[idx], y_tr[idx]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model, loss.item()

model_ok, last_loss = train(make_model())
print(f"correct version: last batch loss {last_loss:.3f}, test accuracy {accuracy(model_ok, X_te, y_te):.3f}")

### Bug 1

**Find the bug. What will happen to the training?**

In [ ]:
def train_bug1(model, epochs=30, bs=50, lr=0.05):
    torch.manual_seed(1)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), bs):
            idx = perm[i:i + bs]
            xb, yb = X_tr[idx], y_tr[idx]
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model, loss.item()

model, last_loss = train_bug1(make_model())
print(f"bug 1: last batch loss {last_loss:.3f}, test accuracy {accuracy(model, X_te, y_te):.3f}")

<details><summary><b>Show the answer</b></summary>

`opt.zero_grad()` is missing. Every `backward()` adds to the gradients of all earlier batches, so the steps get larger and
larger and point in an old direction. The loss explodes (here about 570 for the last batch) and the test accuracy is 0.51 – guessing.
Correct version: loss 0.08, accuracy 0.97.

</details>

### Bug 2

**The model is trained with the correct `train()`. Find the bug in the evaluation. Why do the three numbers differ?**

In [ ]:
def accuracy_bug2(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

model, _ = train(make_model())          # train() leaves the model in training mode
for run in range(3):
    print(f"bug 2: test accuracy {accuracy_bug2(model, X_te, y_te):.3f}")
print(f"correct: test accuracy {accuracy(model, X_te, y_te):.3f}")

<details><summary><b>Show the answer</b></summary>

`model.eval()` is missing. `train()` leaves the model in training mode, so dropout is still active during the evaluation:
every call drops other neurons and gives a different, lower accuracy (here 0.96, 0.95, 0.95 instead of 0.97). `no_grad()` does
not change this (Exercise 2.3).

</details>

### Bug 3

**Find the bug. Hint: look at which labels belong to which inputs.**

In [ ]:
def train_bug3(model, epochs=30, bs=50, lr=0.05):
    torch.manual_seed(1)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), bs):
            xb = X_tr[perm[i:i + bs]]
            yb = y_tr[i:i + bs]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model, loss.item()

model, last_loss = train_bug3(make_model())
print(f"bug 3: last batch loss {last_loss:.3f}, test accuracy {accuracy(model, X_te, y_te):.3f}")

<details><summary><b>Show the answer</b></summary>

The inputs are taken in shuffled order (`perm[i:i + bs]`), but the labels in the original order (`y_tr[i:i + bs]`), so each
input gets the label of a random other point. The model cannot learn the rule: the last loss is 0.693 = ln 2, the loss of guessing
50/50 between two classes, and the test accuracy (0.63) is far below the 0.97 of the correct version. Fix: `yb = y_tr[perm[i:i + bs]]`.

</details>

### Bug 4

**Find the bug. Why can the loss never get close to 0?**

In [ ]:
def train_bug4(model, epochs=30, bs=50, lr=0.05):
    torch.manual_seed(1)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), bs):
            idx = perm[i:i + bs]
            xb, yb = X_tr[idx], y_tr[idx]
            opt.zero_grad()
            probs = torch.softmax(model(xb), dim=1)
            loss = loss_fn(probs, yb)
            loss.backward()
            opt.step()
    return model, loss.item()

model, last_loss = train_bug4(make_model())
print(f"bug 4: last batch loss {last_loss:.3f}, test accuracy {accuracy(model, X_te, y_te):.3f}")
model_ok3, loss3 = train(make_model(), epochs=3)
model_b3, loss_b3 = train_bug4(make_model(), epochs=3)
print(f"after 3 epochs: correct {accuracy(model_ok3, X_te, y_te):.3f}, bug 4 {accuracy(model_b3, X_te, y_te):.3f}")
print("smallest possible loss with this bug:", round(-torch.log_softmax(torch.tensor([1.0, 0.0]), dim=0)[0].item(), 3))

<details><summary><b>Show the answer</b></summary>

`nn.CrossEntropyLoss` expects raw scores (logits) and applies log-softmax itself. Here it gets probabilities, so softmax is
applied twice. Probabilities lie between 0 and 1, so the best the second softmax can do is a score difference of 1: the loss can
never go below −log(e/(e + 1)) ≈ 0.313, and the gradients become small when the model is confident. On this easy problem the
accuracy hardly suffers (0.99 vs 0.97 after 30 epochs, slightly lower after 3), which is why this bug often goes unnoticed – but the
loss values are wrong, and on harder problems learning becomes slower. Fix: pass `model(xb)` directly to the loss.

</details>

### Bug 5

**This evaluation function contains a bug. What happens when it is called?**

In [ ]:
def accuracy_bug5(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(dim=0)
        print("pred shape:", pred.shape, " y shape:", y.shape)
        return (pred == y).float().mean().item()

try:
    print(accuracy_bug5(model_ok, X_te, y_te))
except RuntimeError as e:
    print("RuntimeError:", str(e)[:70])

<details><summary><b>Show the answer</b></summary>

`model(X)` has shape (1000, 2): one row per sample, one column per class. The predicted class is the argmax **over the
classes**, `dim=1`. With `dim=0` you get, for each class, the index of the sample with the highest score: a tensor of
shape (2,), which cannot be compared with the 1,000 labels. (If the shapes happened to match, the code would run and give a
meaningless accuracy – worse than an error.)

</details>

---
## Part 5 – Reading training curves

### Exercise 5.1 – Three learning rates

**Typical patterns.** Learning rate too small: the loss decreases slowly. Too large: the loss oscillates or grows, up to
`inf`/`nan`. Overfitting: the training loss keeps falling while the validation/test loss starts to rise.

A linear regression model (`nn.Linear(1, 1)`, MSE loss, plain gradient descent on all data) is trained with three learning
rates: 0.005, 0.3 and 1.05. The plot shows the three loss curves in random order, labelled A, B, C.
**Which curve belongs to which learning rate?**

In [ ]:
torch.manual_seed(0)
x = torch.randn(200, 1)
y = 3 * x - 1 + 0.3 * torch.randn(200, 1)

def fit(lr, steps=40):
    torch.manual_seed(0)
    lin = nn.Linear(1, 1)
    opt = torch.optim.SGD(lin.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        loss = ((lin(x) - y) ** 2).mean()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

curves = {"A": fit(0.3), "B": fit(1.05), "C": fit(0.005)}
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, losses) in zip(axes, curves.items()):
    ax.plot(losses)
    ax.set_title(f"curve {name}")
    ax.set_xlabel("step")
    ax.set_ylabel("training loss")
plt.tight_layout()
plt.show()
for name, losses in curves.items():
    print(f"curve {name}: first loss {losses[0]:.2f}, last loss {losses[-1]:.2f}")

<details><summary><b>Show the answer</b></summary>

C = 0.005: the loss decreases, but very slowly (still 5.2 after 40 steps). A = 0.3: fast and smooth down to the noise
level (0.10). B = 1.05: the steps are too large, so each step overshoots the minimum further than the last one and the loss grows
(to about 26,000) – with a little more time it becomes `inf`/`nan`. Fix for B: lower the learning rate; for C: increase it.

</details>

### Exercise 5.2 – Overfitting

A large MLP is trained on only 100 noisy training points (10 % of the labels are flipped), without any
regularisation.

**What do the training and test loss curves look like? What would you change?**

In [ ]:
def noisy_data(n, seed):
    X, y = make_data(n, seed)
    g = torch.Generator().manual_seed(seed + 100)
    flip = torch.rand(n, generator=g) < 0.1
    return X, torch.where(flip, 1 - y, y)

Xs_tr, ys_tr = noisy_data(100, seed=3)
Xs_te, ys_te = noisy_data(1000, seed=4)

def run(dropout, weight_decay, epochs=600):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Dropout(dropout),
                          nn.Linear(256, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 2))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    tr, te = [], []
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xs_tr), ys_tr)
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            tr.append(loss_fn(model(Xs_tr), ys_tr).item())
            te.append(loss_fn(model(Xs_te), ys_te).item())
    return tr, te, accuracy(model, Xs_te, ys_te)

tr, te, acc = run(dropout=0.0, weight_decay=0.0)
tr_r, te_r, acc_r = run(dropout=0.5, weight_decay=1e-2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (a, b, title) in zip(axes, [(tr, te, "no regularisation"), (tr_r, te_r, "dropout 0.5 + weight decay")]):
    ax.plot(a, label="training loss"); ax.plot(b, label="test loss")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()
axes[0].set_ylim(0, 1.5)
plt.tight_layout(); plt.show()
best = min(range(len(te)), key=te.__getitem__)
print(f"no regularisation: final train loss {tr[-1]:.3f}, final test loss {te[-1]:.2f}, lowest test loss at epoch {best}, test accuracy {acc:.3f}")
print(f"regularised:       final train loss {tr_r[-1]:.3f}, final test loss {te_r[-1]:.2f}, test accuracy {acc_r:.3f}")

<details><summary><b>Show the answer</b></summary>

Without regularisation the training loss goes to almost 0 (the network memorises all 100 points, including the
flipped labels), while the test loss reaches its minimum early (around epoch 20) and then rises to about 2.5: **overfitting**.
With dropout and weight decay the training loss stays higher (0.3) and the test loss stays low (0.5). Other fixes: early stopping
at the epoch with the lowest validation loss, a smaller network, or more (augmented) data. Note that the test accuracy changes
less (0.79 vs 0.81) than the test loss: the unregularised model is not much more often wrong, but it is very confidently wrong.

</details>